# 🚗 Electric Vehicle (EV) Charging Infrastructure in Germany: Data Analysis & Visualization

*Developed by Khalil Hammod*

This notebook performs an Exploratory Data Analysis (EDA) on the official German EV charging stations dataset. We will clean the data, engineer geographical features, analyze operator distributions, and construct interactive visualizations using **Plotly Express**.

In [ ]:
# Import required packages
import pandas as pd
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

## 📥 1. Loading the Dataset
We load the dataset using a semicolon (`;`) delimiter, which is standard for German CSV exports, and skip any malformed rows.

In [ ]:
# Load the dataset
df = pd.read_csv('rhein-kreis-neuss-ladesaulen-in-deutschland.csv', delimiter=';', on_bad_lines='skip')
df.head()

## 🔍 2. General Data Exploration
Let's examine the shape, data types, and general structure of the dataset.

In [ ]:
print(f"Dataset Shape: {df.shape}")
df.info()

### Missing Values Analysis
Let's look at the distribution of missing values across the features.

In [ ]:
df.isnull().sum()

## 🛠️ 3. Feature Engineering: Extracting Geographical Coordinates
The coordinates in the raw dataset are stored in a single string column `koordinaten` in the format `latitude, longitude`. To plot these points on maps, we need to extract them into separate floating-point `lat` and `long` columns. We also filter out coordinate outliers that lie outside Germany's geographical bounds.

In [ ]:
# Extract latitude and longitude
lat = []
long = []
koord = df['koordinaten'].dropna()

for val in koord.values:
    parts = val.split(',')
    lat.append(float(parts[0].strip()))
    long.append(float(parts[1].strip()))

# Add columns to the DataFrame
df['lat'] = lat
df['long'] = long

# Filter out geographical coordinate outliers outside Germany bounds
# (Typical bounding box: Lat 47.2 to 55.1, Lon 5.8 to 15.1)
df = df[
    (df['lat'] >= 47.2) & (df['lat'] <= 55.1) &
    (df['long'] >= 5.8) & (df['long'] <= 15.1)
]
df[['koordinaten', 'lat', 'long']].head()

## 📊 4. Analyzing EV Charging Operators
Who are the leading providers of charging infrastructure in Germany? Let's check the top 10 operators by the number of charging stations they operate.

In [ ]:
top_operators = df['Betreiber'].value_counts().head(10)
print(top_operators)

# Plot the top operators using Plotly Express
fig_ops = px.bar(
    x=top_operators.index,
    y=top_operators.values,
    labels={'x': 'Operator (Betreiber)', 'y': 'Number of Stations'},
    title='Top 10 EV Charging Station Operators in Germany',
    color=top_operators.values,
    color_continuous_scale='Viridis'
)
fig_ops.update_layout(xaxis_tickangle=-45)
fig_ops.show()

## 🗺️ 5. Geographic Visualization using Plotly Express
Plotly Express provides powerful Mapbox integration. We will explore the geographic distribution of charging stations in specific cities and across operators.

### A. Distribution of Charging Stations in Berlin
Let's visualize the spatial distribution of charging stations in the capital, Berlin, color-coded by the operators.

In [ ]:
berlin_df = df[df['Ort'] == 'Berlin']
fig_berlin = px.scatter_map(
    berlin_df,
    lat="lat",
    lon="long",
    color="Betreiber",
    title="EV Charging Stations in Berlin by Operator",
    zoom=10,
    height=600
)
fig_berlin.update_layout(map_style="carto-darkmatter")
fig_berlin.show()

### B. Distribution of Charging Stations in Bremen
Let's check another major region (Bremen) and add rich hover data (plug types, charging type, and power in kW) to make the map informative.

In [ ]:
bremen_df = df[df['Ort'] == 'Bremen']
fig_bremen = px.scatter_map(
    bremen_df,
    lat="lat",
    lon="long",
    color="Betreiber",
    hover_data=['Art der Ladeeinrichung', 'Steckertypen1', 'P1 [kW]'],
    title="EV Charging Stations in Bremen with Station Details",
    zoom=11,
    height=600
)
fig_bremen.update_layout(map_style="carto-darkmatter")
fig_bremen.show()

### C. Infrastructure of Germany's Top Operator: EnBW
Let's plot the charging stations of the largest operator in Germany, `EnBW mobility+ AG und Co.KG`. To display all points without lagging the browser, we use Mapbox's built-in **clustering** feature.

In [ ]:
enbw_df = df[df['Betreiber'] == 'EnBW mobility+ AG und Co.KG']
fig_enbw = px.scatter_map(
    enbw_df,
    lat="lat",
    lon="long",
    title="EnBW Charging Stations in Germany (Clustered)",
    zoom=5,
    height=700
)
fig_enbw.update_traces(cluster=dict(enabled=True))
fig_enbw.update_layout(map_style="carto-darkmatter")
fig_enbw.show()

## ⚡ 6. Charging Capacity & Connection Types
Let's analyze the electrical properties of the stations, specifically the difference between normal chargers (`Normalladeeinrichtung`) and fast chargers (`Schnellladeeinrichtung`), and the power output distribution.

### A. Ratio of Normal vs. Fast Chargers
Let's see what percentage of the German infrastructure consists of high-speed DC fast chargers.

In [ ]:
charging_types = df['Art der Ladeeinrichung'].value_counts()
fig_pie = px.pie(
    names=charging_types.index,
    values=charging_types.values,
    title="Charging Station Types in Germany (Normal vs. Fast Chargers)",
    hole=0.4,
    color_discrete_sequence=px.colors.qualitative.Pastel
)
fig_pie.show()

### B. Charging Power Output (kW) Analysis
Let's analyze the distribution of rated charging capacity (in kW) across all charging stations in Germany.

In [ ]:
fig_power = px.histogram(
    df,
    x="Nennleistung Ladeeinrichtung [kW]",
    nbins=50,
    title="Distribution of Charging Station Power Ratings (kW) in Germany",
    labels={'Nennleistung Ladeeinrichtung [kW]': 'Power Capacity (kW)'},
    color="Art der Ladeeinrichung",
    color_discrete_map={"Normalladeeinrichtung": "#00CC96", "Schnellladeeinrichtung": "#EF553B"},
    log_y=True
)
fig_power.update_layout(bargap=0.1)
fig_power.show()

## 🔍 7. Data Cleaning: Duplicate Check
Let's analyze duplicates in the dataset. Some stations might be registered multiple times under the same street address, or have identical coordinates.

In [ ]:
total_records = df.shape[0]
unique_coords = df['koordinaten'].nunique()
unique_streets = df['Straße'].nunique()

print(f"Total records in dataset: {total_records}")
print(f"Unique coordinate locations: {unique_coords}")
print(f"Unique streets: {unique_streets}")
print(f"Duplicate coordinate entries: {total_records - unique_coords} ({((total_records - unique_coords)/total_records)*100:.2f}%)")

## 📌 Conclusions & Next Steps
In this notebook, we've cleaned the German EV charging infrastructure data and successfully created interactive maps and analytical charts.

**Key Insights:**
1. **Spatial density**: EV charging stations are highly concentrated around metropolitan areas (Berlin, Hamburg, Munich, Ruhr region).
2. **Type distribution**: Normal AC chargers (`Normalladeeinrichtung`, typically up to 22 kW) make up the vast majority of the network (~80%), while DC fast chargers (`Schnellladeeinrichtung`, 50 kW to 300+ kW) comprise the remaining ~20%.
3. **Dominant Operators**: Large energy providers (like EnBW, E.ON, and local municipal *Stadtwerke*) drive most of the deployment.

**Next Step:**
We will now take this clean dataset and analysis and deploy an interactive **Streamlit Dashboard** (`app.py`) containing sidebar filters for real-time geographic exploration.